# 01 · Análisis exploratorio de datos
**Caso 7 – BI para la logística de última milla**

**Objetivo:** conocer los datos de envíos y comprobar su calidad antes de construir el modelo: valores faltantes (incluida la fecha de entrega vacía en devoluciones), rango de fechas, zonas y transportistas.

**Entrada:** `datos/envios_courier.csv`
**Salida:** ninguna (solo diagnóstico). Las decisiones de limpieza se aplican en `02_etl.ipynb`.

In [1]:
import pandas as pd

RUTA_DATOS = "datos/envios_courier.csv"

df = pd.read_csv(RUTA_DATOS,
                 parse_dates=["fecha_recepcion", "fecha_promesa", "fecha_entrega"])
df.head()

,id_envio,fecha_recepcion,fecha_promesa,fecha_entrega,zona_destino,id_transportista,tipo_servicio,peso_kg,intentos_entrega,estado_final,costo_envio
0,EN000001,2026-01-02,2026-01-03,2026-01-03,Norte,TR05,Express,0.72,1,Entregado,4.25
1,EN000002,2026-01-02,2026-01-05,2026-01-03,Norte,TR02,Estándar,2.31,1,Entregado,2.89
2,EN000003,2026-01-02,2026-01-05,2026-01-04,Sur,TR05,Estándar,1.66,1,Entregado,2.93
3,EN000004,2026-01-02,2026-01-05,2026-01-04,Norte,TR07,Estándar,0.26,1,Entregado,3.41
4,EN000005,2026-01-02,2026-01-03,2026-01-03,Centro,TR08,Express,1.61,1,Entregado,4.29


## 1. Tipos de datos
Las fechas deben ser `datetime64` para poder restarlas; `fecha_entrega` tiene menos valores no nulos que el total de filas.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41952 entries, 0 to 41951
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_envio          41952 non-null  str           
 1   fecha_recepcion   41952 non-null  datetime64[us]
 2   fecha_promesa     41952 non-null  datetime64[us]
 3   fecha_entrega     41239 non-null  datetime64[us]
 4   zona_destino      41952 non-null  str           
 5   id_transportista  41952 non-null  str           
 6   tipo_servicio     41952 non-null  str           
 7   peso_kg           41952 non-null  float64       
 8   intentos_entrega  41952 non-null  int64         
 9   estado_final      41952 non-null  str           
 10  costo_envio       41952 non-null  float64       
dtypes: datetime64[us](3), float64(2), int64(1), str(5)
memory usage: 4.9 MB


## 2. Valores faltantes

In [3]:
df.isna().sum()

id_envio              0
fecha_recepcion       0
fecha_promesa         0
fecha_entrega       713
zona_destino          0
id_transportista      0
tipo_servicio         0
peso_kg               0
intentos_entrega      0
estado_final          0
costo_envio           0
dtype: int64

## 3. ¿Los vacíos de `fecha_entrega` son las devoluciones?

In [4]:
pd.crosstab(df["estado_final"], df["fecha_entrega"].isna(),
            rownames=["Estado"], colnames=["¿Fecha vacía?"])

¿Fecha vacía?,False,True
Estado,,
Devuelto,0,713
Entregado,41239,0


**Conclusión:** los 713 vacíos coinciden 1 a 1 con los envíos devueltos. No es un error de calidad: el envío nunca se entregó.

## 4. Rango de fechas

In [5]:
for col in ["fecha_recepcion", "fecha_promesa", "fecha_entrega"]:
    print(f"{col}: {df[col].min().date()} a {df[col].max().date()}")

fecha_recepcion: 2026-01-02 a 2026-06-30
fecha_promesa: 2026-01-03 a 2026-07-03
fecha_entrega: 2026-01-03 a 2026-07-07


## 5. Zonas, transportistas y demás categorías

In [6]:
for col in ["zona_destino", "id_transportista", "tipo_servicio",
            "estado_final", "intentos_entrega"]:
    print(df[col].value_counts(), "\n")

zona_destino
Norte     11858
Sur       10084
Centro     9912
Valles     5953
Rural      4145
Name: count, dtype: int64 

id_transportista
TR03    5283
TR04    5281
TR07    5252
TR06    5249
TR05    5244
TR02    5231
TR08    5221
TR01    5191
Name: count, dtype: int64 

tipo_servicio
Estándar    29369
Express     12583
Name: count, dtype: int64 

estado_final
Entregado    41239
Devuelto       713
Name: count, dtype: int64 

intentos_entrega
1    34882
2     5018
3     2052
Name: count, dtype: int64 



## 6. Controles de calidad

In [7]:
print("IDs duplicados:", df["id_envio"].duplicated().sum())
print("Entregas antes de la recepción:",
      (df["fecha_entrega"] < df["fecha_recepcion"]).sum())
print("Pesos <= 0:", (df["peso_kg"] <= 0).sum())
df[["peso_kg", "costo_envio"]].describe()

IDs duplicados: 0
Entregas antes de la recepción: 0
Pesos <= 0: 1


,peso_kg,costo_envio
count,41952.000000,41952.000000
mean,2.206407,3.883164
std,1.570934,1.259964
min,0.000000,2.200000
25%,1.060000,2.820000
50%,1.850000,3.710000
75%,2.960000,4.590000
max,17.840000,11.140000


## 7. Regla de la fecha promesa

In [8]:
dias_promesa = (df["fecha_promesa"] - df["fecha_recepcion"]).dt.days
dias_promesa.groupby(df["tipo_servicio"]).value_counts()

tipo_servicio   
Estándar       3    29369
Express        1    12583
Name: count, dtype: int64

## 8. Primer vistazo al problema: puntualidad por zona

In [9]:
ent = df[df["estado_final"] == "Entregado"].copy()
ent["a_tiempo"] = ent["fecha_entrega"] <= ent["fecha_promesa"]
(ent.groupby("zona_destino")["a_tiempo"].mean() * 100).round(1).sort_values()

zona_destino
Rural     19.4
Valles    58.3
Sur       79.9
Centro    86.2
Norte     87.6
Name: a_tiempo, dtype: float64

## Conclusiones del análisis exploratorio
- 41 952 envíos, 11 columnas, del 2 de enero al 30 de junio de 2026.
- Sin IDs duplicados ni fechas imposibles.
- `fecha_entrega` vacía en 713 registros = exactamente los devueltos → el OTD se calcula sobre envíos entregados.
- 1 peso igual a 0 kg → se imputará en el ETL.
- Promesa fija: Estándar 3 días, Express 1 día.
- Primera señal: la puntualidad varía mucho por zona (Rural ≈ 19 %, Norte ≈ 88 %).